## Building an Image Classifier usin PyTorch

In [1]:
import torchvision, torch
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32,scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets",train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets",train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000,5_000])

In [2]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

X_sample, y_sample = train_data[0]
X_sample.shape

torch.Size([1, 28, 28])

In [3]:
X_sample.dtype

torch.float32

FashionMNIST—have a classes attribute containing the list of class names. For example, here’s how we can tell that the sample image represents an ankle boot

In [4]:
train_and_valid_data.classes[y_sample]

'Ankle boot'

In [5]:
from torch import nn
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, hidden_1, hidden_2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, hidden_1), nn.ReLU(),
            nn.Linear(hidden_1, hidden_2), nn.ReLU(),
            nn.Linear(hidden_2, n_classes)
        )
    def forward(self,X):
        return self.mlp(X)

In [6]:
import torchmetrics

device = "cuda" if torch.cuda.is_available else 'cpu'

torch.manual_seed(42)
model = ImageClassifier(n_inputs=28*28,hidden_1=300,hidden_2=100,n_classes=10).to(device)
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.1)
Accuracy = torchmetrics.Accuracy(task='multiclass',num_classes=10).to(device)

c:\Users\TeReX\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def evaluate(model, valid_loader, metric):
    model.eval()
    metric.reset()
    for X_batch, y_batch in valid_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        metric.update(y_pred, y_batch)
    return metric.compute()

def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    history = {'train loss':[], 'train metric':[], 'valid metric':[],}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history['train loss'].append(mean_loss)
        history['train metric'].append(metric.compute().item())
        history['valid metric'].append(evaluate(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train loss'][-1]:.4f}, "
              f"train metric: {history['train metric'][-1]:.4f}, "
              f"valid metric: {history['valid metric'][-1]:.4f}")
    return history

In [8]:
_ = train2(model, optimizer, xentropy, Accuracy, train_loader, valid_loader, 20)

Epoch 1/20, train loss: 0.6058, train metric: 0.7811, valid metric: 0.8428
Epoch 2/20, train loss: 0.4061, train metric: 0.8482, valid metric: 0.8460
Epoch 3/20, train loss: 0.3641, train metric: 0.8642, valid metric: 0.8688
Epoch 4/20, train loss: 0.3340, train metric: 0.8768, valid metric: 0.8712
Epoch 5/20, train loss: 0.3158, train metric: 0.8827, valid metric: 0.8596
Epoch 6/20, train loss: 0.2990, train metric: 0.8877, valid metric: 0.8722
Epoch 7/20, train loss: 0.2868, train metric: 0.8935, valid metric: 0.8746
Epoch 8/20, train loss: 0.2737, train metric: 0.8979, valid metric: 0.8676
Epoch 9/20, train loss: 0.2644, train metric: 0.9010, valid metric: 0.8856
Epoch 10/20, train loss: 0.2535, train metric: 0.9043, valid metric: 0.8806
Epoch 11/20, train loss: 0.2463, train metric: 0.9066, valid metric: 0.8914
Epoch 12/20, train loss: 0.2376, train metric: 0.9106, valid metric: 0.8814
Epoch 13/20, train loss: 0.2295, train metric: 0.9127, valid metric: 0.8868
Epoch 14/20, train lo

In [9]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred = y_pred_logits.argmax(dim=1)
y_pred

tensor([7, 8, 0], device='cuda:0')

In [10]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Bag', 'T-shirt/top']

In [11]:
import torch.nn.functional as F
y_proba = F.softmax(y_pred_logits, dim=1)
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9980, 0.0000,
         0.0020],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000],
        [0.8140, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1860, 0.0000, 0.0000,
         0.0000]], device='cuda:0')

In [12]:
y_top4proba, y_top4indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4proba = F.softmax(y_top4proba, dim=1)
y_top4proba.round(decimals=3)

tensor([[0.9980, 0.0020, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000],
        [0.8140, 0.1860, 0.0000, 0.0000]], device='cuda:0')

In [13]:
y_top4indices

tensor([[7, 9, 3, 8],
        [8, 0, 6, 4],
        [0, 6, 2, 3]], device='cuda:0')

In [14]:
import optuna
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20,300)
    model = ImageClassifier(n_inputs=1*28*28,hidden_1=n_hidden,hidden_2=n_hidden,n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    history = train2(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=10)
    validation_accuracy = max(history["valid metric"])
    return validation_accuracy

In [15]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2026-10-28 22:33:42,837] A new study created in memory with name: no-name-97105878-e27c-4c4d-a6fe-459d27afdb76


Epoch 1/10, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860
Epoch 2/10, train loss: 2.2094, train metric: 0.2799, valid metric: 0.3482
Epoch 3/10, train loss: 2.1164, train metric: 0.4107, valid metric: 0.4516
Epoch 4/10, train loss: 1.9776, train metric: 0.5064, valid metric: 0.5604
Epoch 5/10, train loss: 1.7868, train metric: 0.5865, valid metric: 0.6012
Epoch 6/10, train loss: 1.5776, train metric: 0.6171, valid metric: 0.6236
Epoch 7/10, train loss: 1.3979, train metric: 0.6305, valid metric: 0.6286
Epoch 8/10, train loss: 1.2605, train metric: 0.6353, valid metric: 0.6404
Epoch 9/10, train loss: 1.1572, train metric: 0.6461, valid metric: 0.6412


[I 2026-10-28 22:35:05,458] Trial 0 finished with value: 0.6480000019073486 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.6480000019073486.


Epoch 10/10, train loss: 1.0782, train metric: 0.6541, valid metric: 0.6480
Epoch 1/10, train loss: 1.1610, train metric: 0.6131, valid metric: 0.7376
Epoch 2/10, train loss: 0.6198, train metric: 0.7854, valid metric: 0.8024
Epoch 3/10, train loss: 0.5227, train metric: 0.8188, valid metric: 0.8088
Epoch 4/10, train loss: 0.4815, train metric: 0.8305, valid metric: 0.8282
Epoch 5/10, train loss: 0.4554, train metric: 0.8401, valid metric: 0.8372
Epoch 6/10, train loss: 0.4363, train metric: 0.8464, valid metric: 0.8410
Epoch 7/10, train loss: 0.4207, train metric: 0.8524, valid metric: 0.8458
Epoch 8/10, train loss: 0.4084, train metric: 0.8577, valid metric: 0.8422
Epoch 9/10, train loss: 0.3969, train metric: 0.8610, valid metric: 0.8206


[I 2026-10-28 22:36:41,007] Trial 1 finished with value: 0.8575999736785889 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8575999736785889.


...

Epoch 10/10, train loss: 2.2478, train metric: 0.1017, valid metric: 0.1052
Epoch 1/10, train loss: 2.3063, train metric: 0.0883, valid metric: 0.1048
Epoch 2/10, train loss: 2.3031, train metric: 0.1113, valid metric: 0.1304
Epoch 3/10, train loss: 2.2998, train metric: 0.1336, valid metric: 0.1476
Epoch 4/10, train loss: 2.2966, train metric: 0.1435, valid metric: 0.1506
Epoch 5/10, train loss: 2.2934, train metric: 0.1443, valid metric: 0.1512
Epoch 6/10, train loss: 2.2903, train metric: 0.1435, valid metric: 0.1486
Epoch 7/10, train loss: 2.2871, train metric: 0.1416, valid metric: 0.1456
Epoch 8/10, train loss: 2.2841, train metric: 0.1394, valid metric: 0.1424
Epoch 9/10, train loss: 2.2810, train metric: 0.1372, valid metric: 0.1386


[I 2026-10-28 22:40:30,268] Trial 3 finished with value: 0.15119999647140503 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8575999736785889.


Epoch 10/10, train loss: 2.2780, train metric: 0.1350, valid metric: 0.1368
Epoch 1/10, train loss: 1.8577, train metric: 0.4151, valid metric: 0.6168
Epoch 2/10, train loss: 0.9813, train metric: 0.6644, valid metric: 0.6962
Epoch 3/10, train loss: 0.7642, train metric: 0.7243, valid metric: 0.7414
Epoch 4/10, train loss: 0.6729, train metric: 0.7644, valid metric: 0.7680
Epoch 5/10, train loss: 0.6132, train metric: 0.7885, valid metric: 0.7884
Epoch 6/10, train loss: 0.5712, train metric: 0.8036, valid metric: 0.8038
Epoch 7/10, train loss: 0.5400, train metric: 0.8146, valid metric: 0.8082
Epoch 8/10, train loss: 0.5166, train metric: 0.8217, valid metric: 0.8154
Epoch 9/10, train loss: 0.4986, train metric: 0.8272, valid metric: 0.8212


[I 2026-10-28 22:42:30,367] Trial 4 finished with value: 0.821399986743927 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8575999736785889.


Epoch 10/10, train loss: 0.4843, train metric: 0.8322, valid metric: 0.8214


In [16]:
study.best_params

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}

In [17]:
study.best_value

0.8575999736785889

In [18]:
def objective(trial, train_loader, valid_loader):
    learning_rate = trial.suggest_float('learning_rate',1e-5,1e-1, log=True)
    n_hidden = trial.suggest_int('hidden',20,300)
    model = ImageClassifier(n_inputs=1*28*28,hidden_1=n_hidden,hidden_2=n_hidden,n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    best_validation_accuracy = 0.0
    n_epochs = 1
    for epoch in range(n_epochs):
        history = train2(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs=1)
        validation_accuracy = max(history['valid metric'])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

In [19]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)
study.optimize(objective_with_data, n_trials=5)

[I 2026-10-28 22:42:41,963] Trial 5 finished with value: 0.08540000021457672 and parameters: {'learning_rate': 1.2087541473056957e-05, 'hidden': 292}. Best is trial 1 with value: 0.8575999736785889.


Epoch 1/1, train loss: 2.3016, train metric: 0.0765, valid metric: 0.0854


[I 2026-10-28 22:42:53,902] Trial 6 finished with value: 0.8026000261306763 and parameters: {'learning_rate': 0.021368329072358756, 'hidden': 79}. Best is trial 1 with value: 0.8575999736785889.


Epoch 1/1, train loss: 0.8836, train metric: 0.6906, valid metric: 0.8026


[I 2026-10-28 22:43:05,201] Trial 7 pruned. 


Epoch 1/1, train loss: 2.3066, train metric: 0.0995, valid metric: 0.1062


[I 2026-10-28 22:43:17,524] Trial 8 pruned. 


Epoch 1/1, train loss: 2.2888, train metric: 0.1254, valid metric: 0.1458


[I 2026-10-28 22:43:29,733] Trial 9 pruned. 


Epoch 1/1, train loss: 2.2618, train metric: 0.1202, valid metric: 0.1706


In [20]:
from functools import partial

objective_with_data = partial(objective, train_loader=train_loader, valid_loader=valid_loader)

In [21]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler,
                            pruner=pruner)
study.optimize(objective_with_data, n_trials=20)

[I 2026-10-28 22:43:29,751] A new study created in memory with name: no-name-55d54106-8743-433c-9e45-43a6e9f479a7
[I 2026-10-28 22:43:40,875] Trial 0 finished with value: 0.1860000044107437 and parameters: {'learning_rate': 0.00031489116479568613, 'hidden': 287}. Best is trial 0 with value: 0.1860000044107437.


Epoch 1/1, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1860


[I 2026-10-28 22:43:52,436] Trial 1 finished with value: 0.7379999756813049 and parameters: {'learning_rate': 0.008471801418819975, 'hidden': 188}. Best is trial 1 with value: 0.7379999756813049.


Epoch 1/1, train loss: 1.1586, train metric: 0.6173, valid metric: 0.7380


[I 2026-10-28 22:44:03,463] Trial 2 finished with value: 0.10000000149011612 and parameters: {'learning_rate': 4.207988669606632e-05, 'hidden': 63}. Best is trial 1 with value: 0.7379999756813049.


...

Epoch 1/1, train loss: 1.8640, train metric: 0.4273, valid metric: 0.6050


[I 2026-10-28 22:47:11,943] Trial 18 pruned. 


Epoch 1/1, train loss: 1.2253, train metric: 0.6046, valid metric: 0.7052


[I 2026-10-28 22:47:25,118] Trial 19 finished with value: 0.7897999882698059 and parameters: {'learning_rate': 0.07981669924644823, 'hidden': 85}. Best is trial 11 with value: 0.828000009059906.


Epoch 1/1, train loss: 0.6326, train metric: 0.7688, valid metric: 0.7898


In [22]:
study.best_params


{'learning_rate': 0.05590458715170972, 'hidden': 70}

In [23]:
study.best_value

0.828000009059906

In [24]:
torch.save(model, "my_fashion_mnist.pt")

In [25]:
loaded_model = torch.load("my_fashion_mnist.pt", weights_only=False)

In [26]:
loaded_model.eval()
y_pred_logits = loaded_model(X_new)

In [27]:
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")

In [33]:
type(model.state_dict())

collections.OrderedDict

In [31]:
new_model = ImageClassifier(n_inputs=1 * 28 * 28, hidden_1=300, hidden_2=100,
                            n_classes=10)
loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)
new_model.load_state_dict(loaded_weights)
new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

In [36]:
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {
        "n_inputs": 1 * 28 * 28,
        "hidden_1": 300,
        "hidden_2": 100,
        "n_classes": 10,
    }
}
torch.save(model_data, "my_fashion_mnist_model.pt")

In [37]:
loaded_data = torch.load("my_fashion_mnist_model.pt", weights_only=True)
new_model = ImageClassifier(**loaded_data["model_hyperparameters"])
new_model.load_state_dict(loaded_data["model_state_dict"])
new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

In [39]:
torchscript_model = torch.jit.trace(model, X_new)
torchscript_model = torch.jit.script(model)
optimized_model = torch.jit.optimize_for_inference(torchscript_model)
optimized_model.save("my_fashion_mnist_torchscript.pt")
loaded_torchscript_model = torch.jit.load("my_fashion_mnist_torchscript.pt")
y_pred_logits = loaded_torchscript_model(X_new)
y_pred_logits

tensor([[ -2.7129,  -1.5324,  -5.4277,   1.0536,  -2.5764,  -2.0503,  -2.6520,
          11.3767,   0.9154,   4.9877],
        [ 10.0055, -10.9136,  -2.0204,  -5.1943,   3.8678,  -9.9683,   4.5832,
          -2.1603,  20.0951,  -4.7132],
        [ 10.2820,  -4.5995,   2.0113,   1.9588,  -1.7215,  -0.2499,   8.8040,
          -7.6392,  -0.9526,  -6.5431]], device='cuda:0')

In [40]:
compiled_model = torch.compile(model)